In [1]:
!pip install pandas numpy scikit-learn tensorflow

#### 1. Load Data

In [6]:
import pandas as pd
df = pd.read_csv('imbalanced_data.csv')  
df = df[['tweet', 'label']]

print(df.head())

                                               tweet  label
0   @user when a father is dysfunctional and is s...      0
1  @user @user thanks for #lyft credit i can't us...      0
2                                bihday your majesty      0
3  #model   i love u take with u all the time in ...      0
4             factsguide: society now    #motivation      0


In [7]:
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#### 2. Text Cleaning

In [8]:
import re
def clean_text(text):
    text = text.lower()                         
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # remove special chars & numbers
    text = re.sub(r'\s+', ' ', text).strip()     # remove extra spaces
    return text

df['tweet'] = df['tweet'].apply(clean_text)
df = df[df['tweet'].str.strip() != '']           # drop any now-empty tweets

#### 3. Tokenization & Vocabulary

In [9]:
from collections import Counter

VOCAB_SIZE = 5000
PAD_TOKEN = 0
OOV_TOKEN = 1

# Count all words
all_words = []
for tweet in df['tweet']:
    all_words.extend(tweet.split())

# Take top 5000 most common words
most_common = Counter(all_words).most_common(VOCAB_SIZE - 2)  # -2 for PAD and OOV

# Build vocab: word -> index (PAD=0, OOV=1, rest start from 2)
vocab = {'<PAD>': 0, '<OOV>': 1}
for idx, (word, _) in enumerate(most_common, start=2):
    vocab[word] = idx

#### 4. Text to Sequences + Padding

In [10]:
MAX_LEN = 30

def text_to_sequence(text, vocab):
    return [vocab.get(word, OOV_TOKEN) for word in text.split()]

def pad_sequence(seq, max_len):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [PAD_TOKEN] * (max_len - len(seq))

import numpy as np

X = np.array([pad_sequence(text_to_sequence(t, vocab), MAX_LEN) for t in df['tweet']])
y = np.array(df['label'])

#### 5. Train/Test Split (Stratified)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify is KEY for imbalanced data
)

#### 6. Handle Class Imbalance

In [12]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
# This will give roughly {0: 0.54, 1: 7.13} — penalizes missing hate speech more

#### 7. Build the Model

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

EMBED_DIM = 64
RNN_UNITS = 64

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SimpleRNN(RNN_UNITS),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\sadan\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### 8.Train

In [14]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict   # handles imbalance
)

Epoch 1/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.7326 - loss: 0.5471 - val_accuracy: 0.5158 - val_loss: 0.7996
Epoch 2/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8702 - loss: 0.3112 - val_accuracy: 0.6660 - val_loss: 0.8859
Epoch 3/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9146 - loss: 0.1957 - val_accuracy: 0.8537 - val_loss: 0.4209
Epoch 4/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9394 - loss: 0.1493 - val_accuracy: 0.9011 - val_loss: 0.3497
Epoch 5/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.9554 - loss: 0.1170 - val_accuracy: 0.8623 - val_loss: 0.4449
Epoch 6/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9571 - loss: 0.1017 - val_accuracy: 0.8956 - val_loss: 0.3249
Epoch 7/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9646 - loss: 0.0874 - val_accuracy: 0.9011 - val_loss: 0.3394
Epoch 8/10
720/720 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9720 - loss: 0.0726 - 

#### 9. Evaluate

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Test Accuracy:", (y_pred == y_test).mean())
print(classification_report(y_test, y_pred, target_names=['Non-Hate', 'Hate']))
print(confusion_matrix(y_test, y_pred))

200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step
Test Accuracy: 0.9195869837296621
              precision    recall  f1-score   support

    Non-Hate       0.97      0.94      0.96      5944
        Hate       0.44      0.59      0.51       448

    accuracy                           0.92      6392
   macro avg       0.71      0.77      0.73      6392
weighted avg       0.93      0.92      0.92      6392

[[5615  329]
 [ 185  263]]
